In [4]:
import pandas as pd

# 读取 Excel 文件
FARE = pd.read_excel("../Data/BUS/SECTION_FARE.xlsx", engine="openpyxl")  

# 查看前几行数据
print(FARE.head())


   ROUTE_ID  ROUTE_SEQ  ON_SEQ  OFF_SEQ  PRICE LAST_UPDATE_DATE
0      1001          1       1        2    6.7       2025-01-02
1      1001          1       1        3    6.7       2025-01-02
2      1001          1       1        4    6.7       2025-01-02
3      1001          1       1        5    6.7       2025-01-02
4      1001          1       1        6    6.7       2025-01-02


In [5]:
ROUTE = pd.read_excel("..\Data\BUS\ROUTE.xlsx", engine="openpyxl")  

# 查看前几行数据
print(ROUTE.head())


   ROUTE_ID COMPANY_CODE  DISTRICT ROUTE_NAMEC ROUTE_NAMES ROUTE_NAMEE  \
0      1001          KMB       NaN           1           1           1   
1      1002          KMB       NaN          10          10          10   
2      1006      KMB+CTB       NaN        102P        102P        102P   
3      1008      KMB+CTB       NaN         103         103         103   
4      1010      KMB+CTB       NaN         106         106         106   

   ROUTE_TYPE SERVICE_MODE  SPECIAL_TYPE  JOURNEY_TIME  ... LOC_START_NAMES  \
0           1            R             0            40  ...             竹园邨   
1           1            R             0            55  ...              彩云   
2           1            T             1            64  ...             筲箕湾   
3           1            R             0            59  ...             竹园邨   
4           1            R             0            82  ...             黄大仙   

    LOC_START_NAMEE LOC_END_NAMEC LOC_END_NAMES                LOC_END_NAMEE  \


In [7]:
RSTOP = pd.read_excel("..\Data\BUS\ROUTE_STOP.xlsx", engine="openpyxl")  

# 查看前几行数据
print(RSTOP.head())


   ROUTE_ID  ROUTE_SEQ  STOP_SEQ  STOP_ID  STOP_PICK_DROP STOP_NAMEC  \
0      1001          1         1     4001               2      竹園邨總站   
1      1001          1         2     4002               3       天虹小學   
2      1001          1         3     4003               3     馬仔坑遊樂場   
3      1001          1         4     4004               3       摩士公園   
4      1001          1         5     4005               3    摩士公園體育館   

  STOP_NAMES                      STOP_NAMEE LAST_UPDATE_DATE  
0      竹园邨总站   CHUK YUEN ESTATE BUS TERMINUS       2023-02-21  
1       天虹小学          RAINBOW PRIMARY SCHOOL       2023-02-21  
2     马仔坑游乐场  MA CHAI HANG RECREATION GROUND       2023-02-21  
3       摩士公园                      MORSE PARK       2023-02-21  
4    摩士公园体育馆        MORSE PARK SPORTS CENTRE       2023-02-21  


In [8]:
STOP = pd.read_excel("..\Data\BUS\STOP_XY.xlsx", engine="openpyxl")  

# 查看前几行数据
print(STOP.head())


   STOP_ID  STOP_TYPE       X       Y LAST_UPDATE_DATE  STOP_CODE
0        2          1  843622  813951       2024-02-02        NaN
1        3          1  843333  814109       2023-03-04        NaN
2        4          1  842947  814013       2023-03-04        NaN
3        5          1  842646  813814       2023-03-04        NaN
4        6          1  842419  813736       2023-03-04        NaN


In [12]:
import pandas as pd

# 假设已经加载了数据：
# FARE = pd.read_csv("fare.csv")
# ROUTE = pd.read_csv("route.csv")
# RSTOP = pd.read_csv("rstop.csv")

# 1. 关联 ROUTE 表，添加 ROUTE_NAMES
fare_with_route = FARE.merge(ROUTE[['ROUTE_ID', 'ROUTE_NAMES','COMPANY_CODE']], on='ROUTE_ID', how='left')

# 2. 关联 RSTOP 表，获取 ON_SEQ_NAMES 和 ON_SEQ_STOP_ID
fare_with_on_seq = fare_with_route.merge(
    RSTOP[['ROUTE_ID', 'ROUTE_SEQ', 'STOP_SEQ', 'STOP_NAMES', 'STOP_ID']],
    left_on=['ROUTE_ID', 'ROUTE_SEQ', 'ON_SEQ'],
    right_on=['ROUTE_ID', 'ROUTE_SEQ', 'STOP_SEQ'],
    how='left'
).rename(columns={'STOP_NAMEC': 'ON_SEQ_NAMES', 'STOP_ID': 'ON_SEQ_STOP_ID'})

# 3. 关联 RSTOP 表，获取 OFF_SEQ_NAMES 和 OFF_SEQ_STOP_ID
fare_with_off_seq = fare_with_on_seq.merge(
    RSTOP[['ROUTE_ID', 'ROUTE_SEQ', 'STOP_SEQ', 'STOP_NAMES', 'STOP_ID']],
    left_on=['ROUTE_ID', 'ROUTE_SEQ', 'OFF_SEQ'],
    right_on=['ROUTE_ID', 'ROUTE_SEQ', 'STOP_SEQ'],
    how='left'
).rename(columns={'STOP_NAMEC': 'OFF_SEQ_NAMES', 'STOP_ID': 'OFF_SEQ_STOP_ID'})


fare_with_on_XY=fare_with_off_seq.merge(
    STOP[['STOP_ID','X','Y']],
    left_on=['ON_SEQ_STOP_ID'],
    right_on=['STOP_ID'],
    how='left'
).rename(columns={'X':'ON_SEQ_STOP_X','Y':'ON_SEQ_STOP_Y'})

fare_with_off_XY=fare_with_on_XY.merge(
    STOP[['STOP_ID','X','Y']],
    left_on=['OFF_SEQ_STOP_ID'],
    right_on=['STOP_ID'],
    how='left'
).rename(columns={'X':'OFF_SEQ_STOP_X','Y':'OFF_SEQ_STOP_Y'})

# 5. 查看结果
print(fare_with_off_XY.head())

# 6. 将结果存入新的 DataFrame
final_fare_df = fare_with_off_XY




   ROUTE_ID  ROUTE_SEQ  ON_SEQ  OFF_SEQ  PRICE LAST_UPDATE_DATE ROUTE_NAMES  \
0      1001          1       1        2    6.7       2025-01-02           1   
1      1001          1       1        3    6.7       2025-01-02           1   
2      1001          1       1        4    6.7       2025-01-02           1   
3      1001          1       1        5    6.7       2025-01-02           1   
4      1001          1       1        6    6.7       2025-01-02           1   

  COMPANY_CODE  STOP_SEQ_x STOP_NAMES_x  ON_SEQ_STOP_ID  STOP_SEQ_y  \
0          KMB           1        竹园邨总站            4001           2   
1          KMB           1        竹园邨总站            4001           3   
2          KMB           1        竹园邨总站            4001           4   
3          KMB           1        竹园邨总站            4001           5   
4          KMB           1        竹园邨总站            4001           6   

  STOP_NAMES_y  OFF_SEQ_STOP_ID  STOP_ID_x  ON_SEQ_STOP_X  ON_SEQ_STOP_Y  \
0         天虹小学        

In [ ]:
final_fare_df.to_excel('../Data/BUS/Converted_Fare.xlsx', index=False, engine='openpyxl')